In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import zarr
import dask
from dask.diagnostics import ProgressBar

In [ ]:
zarr_path_parent = "/mnt/tier2/project/p200177/u101834/DE371_bis/MEPS_subdomain/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_newgrid.zarr"
zarr_path_parent = "/project/home/p200177/u101329/DE371_bis/MEPS_subdomain/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced.zarr" # MEluxina
zarr_path_parent = "/ec/res4/hpcperm/smcd/data/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced.zarr" # Atos 

In [ ]:
ds = xr.open_zarr(zarr_path_parent, zarr_format=2)


In [ ]:
print(ds)


In [ ]:
def latlon_to_harmonie_grid(
    lon, lat,
    NLON=960,
    NLAT=1080,
    LONC=17.5,
    LATC=63.3,
    LON0=15.0,
    LAT0=63.3,
    GSIZE=2500.0,
    R=6371229
):
    """
    Convert lat/lon (degrees, 1D arrays) to HARMONIE grid indices (i, j)
    using Lambert Conformal Conic (1SP).
    """

    # --- Degrees to radians
    lon = np.deg2rad(lon)
    lat = np.deg2rad(lat)
    lon0 = np.deg2rad(LON0)
    lat0 = np.deg2rad(LAT0)
    lonc = np.deg2rad(LONC)
    latc = np.deg2rad(LATC)

    # --- LCC constants (1 standard parallel: LAT0)
    n = np.sin(lat0)
    F = (np.cos(lat0) * np.tan(np.pi / 4 + lat0 / 2) ** n) / n

    def rho(phi):
        return R * F / np.tan(np.pi / 4 + phi / 2) ** n

    # --- Projection of input points
    rho_p = rho(lat)
    theta = n * (lon - lon0)

    x = rho_p * np.sin(theta)
    y = rho(lat0) - rho_p * np.cos(theta)

    # --- Projection of grid center
    rho_c = rho(latc)
    theta_c = n * (lonc - lon0)

    xc = rho_c * np.sin(theta_c)
    yc = rho(lat0) - rho_c * np.cos(theta_c)

    # --- Convert meters → grid indices
    i = (x - xc) / GSIZE + (NLON - 1) / 2
    j = (y - yc) / GSIZE + (NLAT - 1) / 2

    return i, j


In [ ]:
ij = latlon_to_harmonie_grid(ds["longitudes"],
    ds["latitudes"])


In [ ]:
i = ij[0].compute()
j = ij[1].compute()
print(i.min(), i.max())
print(j.min(), j.max())

In [ ]:
i

In [ ]:
j

In [ ]:
inc = np.array(range(0,ds.dims['cell']))
plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=inc,
    s=1
)
plt.colorbar(label="deg")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Lon")
plt.show()


In [ ]:

plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=ds["longitudes"],
    s=1
)
plt.colorbar(label="deg")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Lon")
plt.show()



In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=ds["latitudes"],
    s=1
)
plt.colorbar(label="deg")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Lat")
plt.show()

In [ ]:
ifrac = i - np.floor(i)

In [ ]:
print(ifrac.max(),ifrac.min())

plt.hist(ifrac,bins=100)

In [ ]:
jfrac = j - np.floor(j)

In [ ]:
print(jfrac.max(),jfrac.min())

plt.hist(ifrac,bins=100)

In [ ]:
ii = np.ceil(i.data).astype('int') 
jj = np.ceil(j.data).astype('int')

In [ ]:
print(np.min(ii), np.max(ii))

In [ ]:
print(np.min(jj), np.max(jj))

In [ ]:
largest = np.zeros((577-292+1,530-219+1)).astype('bool')
print(largest.shape)

In [ ]:
for i in range(0,ds.dims['cell']):
    largest[jj[i]-292,ii[i]-219] = True

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(largest)
plt.show()

In [ ]:
ff = np.where(  jj>= 300+256,  True, False)
ffi = ii[ff]

print(np.min(ffi), np.max(ffi))

In [ ]:
ff = np.where(  ii>= 221,  True, False)
ffj = jj[ff]

print(np.min(ffj), np.max(ffj))

In [ ]:
mii = np.where( np.logical_and(ii>=(255), ii<(255+256)), True, False)

In [ ]:
mjj = np.where( np.logical_and(jj>=305 , jj<305+256), True, False)



In [ ]:
mask = np.logical_and(mii,mjj)

In [ ]:
indixes = np.array(range(0,ds.dims['cell']))

ffilter  = indixes[mask]

In [ ]:
ffilter


In [ ]:
ds_sub = ds.isel(cell=ffilter)

In [ ]:
print("Selected cells:", ffilter.size)
print(ds_sub)

In [ ]:
output_path = '/project/home/p200177/u101329/DE371_bis/MEPS_subdomain'

In [ ]:
with ProgressBar():
    ds_sub.to_zarr(
        f"{output_path}/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain.zarr",
        mode="w",
        zarr_format=2,
    )